# 02 — Pre-training, training, and post-training

**Purpose:** validate the data and U-Net ResNet34 v2 configuration, then explicitly run or resume training and inspect trained checkpoints.  
**Inputs:** MAGFiLO train JPEGs/COCO JSON, `configs/unet_resnet34_v2.yaml`, optional existing checkpoints.  
**Outputs:** displayed diagnostics; opt-in smoke checkpoints under `smoke_runs/`; opt-in production `latest.pth` and `best_model.pth`.  
**Dependencies:** project `.conda`, PyTorch, OpenCV, Albumentations, `src/filament_seg`.  
**Execution:** run top to bottom from root or `notebooks/`. All training and extended evaluation flags default to `False`.  
**Runtime:** preflight is seconds; a smoke check is brief; full training can take hours.  
**Next:** `04_instance_error_analysis.ipynb`, `05_postprocessing_optimization.ipynb`, then `03_submission.ipynb`.

Both v1 and v2 used ImageNet-pretrained ResNet34 encoders in their historical training runs. Loading a pretrained encoder is initialization, not a trained segmentation checkpoint.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src' / 'filament_seg' / 'dataset.py').is_file() else cwd.parent
assert (project_root / 'src' / 'filament_seg' / 'dataset.py').is_file(), cwd
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from filament_seg.utils import set_seed, load_config, get_device, get_logger
from filament_seg.model import build_model
from filament_seg.losses import BCEDiceLoss
from filament_seg.evaluate import compute_dice, compute_iou, evaluate_full_resolution_dataset, compare_models
from filament_seg.dataset import SolarFilamentDataset
from filament_seg.preprocessing import ForegroundAwareCrop, get_train_transform, get_val_transform
from filament_seg.train import load_checkpoint, save_checkpoint, train_v2_model, validate_one_epoch

# Load Configuration & Set Seed
config_path = project_root / 'configs' / 'unet_resnet34_v2.yaml'
config = load_config(config_path if config_path.is_file() else project_root / 'configs' / 'baseline.yaml')
set_seed(config['project']['seed'])
device = get_device(config['training'].get('device'))
print(f'Python: {sys.executable}')
print(f'Project root: {project_root}')
print(f'Device selected: {device}')
print(f'Experiment: {config["project"]["experiment_name"]}')
assert Path(sys.executable).resolve() == (project_root / '.conda' / 'python.exe').resolve(), sys.executable


## 1. Dataset & DataLoader Setup (Deterministic 565/142 Physical Split)

Split 707 physical filenames once (seed 42, 80/20: 565 train / 142 val) ensuring zero leakage across splits.
- Training dataset uses foreground-aware crop (512x512, foreground_prob=0.7).
- Validation dataset uses center crop for epoch-level loss tracking.
- Evaluation dataset preserves full 2048x2048 resolution for competition-relevant evaluation.

In [ ]:
train_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026' / 'train'
image_dir = train_dir / 'train_images'
annotation_path = train_dir / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'

train_dataset = SolarFilamentDataset(image_dir, annotation_path, transform=get_train_transform(512))
val_dataset = SolarFilamentDataset(image_dir, annotation_path, transform=get_val_transform(512))
val_eval_dataset = SolarFilamentDataset(image_dir, annotation_path)
assert len(train_dataset) == len(val_dataset) == len(val_eval_dataset) == 707

indices = np.random.default_rng(config['project']['seed']).permutation(len(train_dataset))
train_indices = indices[:565].tolist()
val_indices = indices[565:].tolist()

train_names = {train_dataset.filenames[i] for i in train_indices}
val_names = {val_dataset.filenames[i] for i in val_indices}
assert len(train_names) == 565 and len(val_names) == 142
assert train_names.isdisjoint(val_names)

train_loader = DataLoader(Subset(train_dataset, train_indices), batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(Subset(val_dataset, val_indices), batch_size=4, shuffle=False, num_workers=0)
print(f'Train images: {len(train_indices)}, batches: {len(train_loader)}')
print(f'Validation images: {len(val_indices)}, batches: {len(val_loader)}')

## A. Pre-training readiness

Check the physical split, annotation alternatives, normalized image/mask pairs, CUDA memory, architecture, loss, scheduler, and checkpoint location before enabling any optimization.

In [ ]:
assert len(train_indices)==565 and len(val_indices)==142 and train_names.isdisjoint(val_names)
assert all(len(train_dataset.images_by_filename[name]) >= 1 for name in train_names | val_names)
probe_batch = next(iter(train_loader))
assert probe_batch['image'].shape[1:] == probe_batch['mask'].shape[1:] == (1,512,512)
assert 0 <= float(probe_batch['image'].min()) <= float(probe_batch['image'].max()) <= 1
assert set(torch.unique(probe_batch['mask']).tolist()) <= {0.,1.}
print('Preflight batch:', tuple(probe_batch['image'].shape), 'foreground fraction:', float(probe_batch['mask'].mean()))
print('CUDA:', torch.cuda.is_available(), 'device:', device)
if device.type == 'cuda': print('GPU:', torch.cuda.get_device_name(device), 'MiB:', torch.cuda.get_device_properties(device).total_memory/2**20)
print('Checkpoint directory:', project_root / config['training']['checkpoint_dir'])

## 2. CUDA Model Initialization & Pretrained ResNet34 Encoder

Initialize U-Net with ImageNet-pretrained ResNet34 encoder adapted to single-channel grayscale input on CUDA.

In [ ]:
pretrained = config['model'].get('encoder_weights') == 'imagenet'
model = build_model(pretrained=pretrained, in_channels=1, classes=1).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model Backbone: {config["model"]["encoder_name"]} (Pretrained: {pretrained})')
print(f'Device: {next(model.parameters()).device}')
print(f'Total parameters: {total_params:,} | Trainable: {trainable_params:,}')

## 3. v2 Loss, Optimizer & Scheduler Configuration

Configure calibrated positive loss weighting (`pos_weight=8.0`), AdamW optimizer, and CosineAnnealingLR scheduler.

In [ ]:
train_cfg = config['training']
loss_cfg = train_cfg.get('loss', {})
criterion = BCEDiceLoss(
    bce_weight=float(loss_cfg.get('bce_weight', 0.5)),
    dice_weight=float(loss_cfg.get('dice_weight', 0.5)),
    smooth=float(loss_cfg.get('smooth', 1.0)),
    pos_weight=float(loss_cfg.get('pos_weight', 8.0)),
)
lr = float(train_cfg.get('learning_rate', 3e-4))
wd = float(train_cfg.get('weight_decay', 1e-4))
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
sched_cfg = train_cfg.get('scheduler', {})
t_max = int(sched_cfg.get('t_max', train_cfg.get('epochs', 30)))
eta_min = float(sched_cfg.get('eta_min', 1e-6))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max, eta_min=eta_min)
print(f'BCEDiceLoss configured: pos_weight={loss_cfg.get("pos_weight", 8.0)}')
print(f'Optimizer: AdamW (lr={lr}, weight_decay={wd})')
print(f'Scheduler: CosineAnnealingLR (T_max={t_max}, eta_min={eta_min})')

## 4. AMP Setup & Gradient Clipping

Configure automatic mixed precision (AMP) with PyTorch GradScaler and gradient norm clipping (`max_norm=1.0`) to prevent gradient explosion and inf checks errors.

In [ ]:
use_amp = config['training'].get('amp', True) and device.type == 'cuda'
scaler = torch.amp.GradScaler(device='cuda', enabled=use_amp)
clip_grad = float(config['training'].get('clip_grad_norm', 1.0))
print(f'AMP enabled: {use_amp}')
print(f'GradScaler initialized for device: {device.type}')
print(f'Gradient norm clipping threshold: {clip_grad}')

## B. Training — optional smoke test

Execute a 2-batch smoke test to verify CUDA execution, AMP autocast, gradient flow, loss finiteness, checkpoint saving to `checkpoints/unet_resnet34_v2/smoke_checkpoint.pth`, and resumption.

In [ ]:
RUN_SMOKE_TEST = False  # Explicitly enable a new, uniquely named smoke artifact.
if RUN_SMOKE_TEST:
    smoke_result = train_v2_model(config, train_loader, val_loader, val_dataset, val_indices, device=device, smoke_test=True)
    print('Smoke result:', smoke_result)
else:
    print('Smoke test skipped. Existing smoke and production checkpoints are untouched.')

## B. Training — optional full run and safe resume

**Ready for manual execution:** Running the cell below trains the v2 model for 30 epochs (or resumes from `checkpoints/unet_resnet34_v2/latest.pth` if interrupted).
To execute full training manually, set `RUN_FULL_TRAINING = True`.

In [ ]:
RUN_FULL_TRAINING = False  # Explicit opt-in; never runs during Run All by default.
if RUN_FULL_TRAINING:
    train_result = train_v2_model(config, train_loader, val_loader, val_dataset, val_indices, device=device, resume=True)
    print('Training status:', train_result.get('status', 'run_finished'))
    print('Latest:', train_result['latest_checkpoint'], 'Best:', train_result['best_checkpoint'])
else:
    print('Full training skipped.')

## 7. Checkpoint Resumption Logic & Verification

Demonstrate checkpoint restoration logic:
- Full training with `resume=True` checks ONLY `checkpoints/unet_resnet34_v2/latest.pth`.
- If `latest.pth` exists, it restores model weights, optimizer, scheduler, scaler, epoch, and validation history.
- If `latest.pth` does not exist, training initializes cleanly from ImageNet-pretrained ResNet34 weights (Epoch 1).
- `smoke_checkpoint.pth` is strictly isolated and never used as an automatic resume source.

In [ ]:
v2_latest_path = project_root / config['training']['checkpoint_dir'] / 'latest.pth'
v2_best_path = project_root / config['training']['checkpoint_dir'] / 'best_model.pth'
if v2_latest_path.is_file():
    state = torch.load(v2_latest_path, map_location='cpu', weights_only=False)  # trusted local file
    print('Latest checkpoint:', v2_latest_path, 'epoch:', state.get('epoch'), 'history entries:', len(state.get('history',[])))
    print('Next epoch:', state.get('epoch',0)+1, 'configured maximum:', config['training']['epochs'])
    print('Best metric:', state.get('best_val_dice'), 'purpose:', state.get('purpose','legacy full training'))
else:
    print('No production latest checkpoint; smoke weights are never used for resume.')
print('Best checkpoint exists:', v2_best_path.is_file())

## 8. Full-Resolution Validation

Perform full-resolution 2048x2048 validation using tiled prediction with 2D Hann window blending, calculating semantic Dice, IoU, Precision, Recall, and Panoptic Quality.

In [ ]:
RUN_POST_TRAINING_EVAL = False  # Set True for bounded 5-image native-resolution evaluation.
if RUN_POST_TRAINING_EVAL:
    if not v2_best_path.is_file(): raise FileNotFoundError(v2_best_path)
    trained_model = build_model(pretrained=False).to(device)
    load_checkpoint(v2_best_path, trained_model, map_location=device)
    val_results = evaluate_full_resolution_dataset(trained_model, val_eval_dataset, val_indices[:5], device=device, threshold=0.75, min_area=300, tile_size=512, overlap=128)
    print(val_results['summary'])
else:
    print('Post-training inference skipped; no initialized encoder is substituted for trained weights.')

## 9. Baseline v1 versus v2 Model Comparison

Evaluates and compares the original baseline (`checkpoints/unet_resnet34/best_model.pth`) and v2 (`checkpoints/unet_resnet34_v2/best_model.pth` or initialized v2) on the exact same validation images using full-resolution tiled inference.

In [ ]:
RUN_MODEL_COMPARISON = False  # Opt in after verifying both trained checkpoints.
if RUN_MODEL_COMPARISON:
    v1_path = project_root / 'checkpoints/unet_resnet34/best_model.pth'
    if not v1_path.is_file() or not v2_best_path.is_file(): raise FileNotFoundError('Both trained checkpoints are required')
    model_v1 = build_model(pretrained=False).to(device)
    model_v2 = build_model(pretrained=False).to(device)
    load_checkpoint(v1_path, model_v1, map_location=device)
    load_checkpoint(v2_best_path, model_v2, map_location=device)
    result = compare_models(model_v1, model_v2, val_eval_dataset, val_indices[:5], device=device,
                            threshold_a=0.75, min_area_a=300, threshold_b=0.75, min_area_b=300,
                            model_a_name='v1 trained', model_b_name='v2 trained')
    display(result['comparison_table'])
else:
    print('Model comparison skipped.')

## C. Post-training history and limitations

The plots below use recorded checkpoint metadata only. Patch Dice selects the current checkpoint; diagnostic full-resolution PQ can disagree with that choice. Use notebook 04 for instance errors and notebook 05 for controlled tuning. The local matcher has not been proven identical to the organizer's scorer.

In [ ]:
if v2_latest_path.is_file():
    history = state.get('history', [])
    if history:
        import pandas as pd
        frame = pd.DataFrame(history)
        print(frame.tail(3).to_string(index=False))
        keys = [key for key in ['train_loss','val_loss','val_dice','val_iou','lr'] if key in frame]
        frame.plot(x='epoch', y=keys, subplots=True, figsize=(8, 2*len(keys)), title=keys)
        plt.tight_layout(); plt.show()
else:
    print('No trained checkpoint history available.')